# 모델 배포 개론 07 (v2) — [프로젝트 2] 트랜스포머 & RAG 챗봇 서비스 고도화
Last modified : 2026.08  
작성 : 박광석 (모두의연구소)  
수정 : 김지성, 박기웅 (모두의연구소)
> **🚀 v2 핵심 개선 및 추가 기능**
> 1. **안전한 컨텍스트 슬라이딩 윈도우**: 턴(`[User, Assistant]`) 단위 보존 삭제로 프롬프트 붕괴 버그 해결
> 2. **실시간 토큰 스트리밍 (Chunked Transfer)**: `TextIteratorStreamer` & `StreamingResponse` 도입 (TTFT 단축)
> 3. **ChromaDB 기반 RAG(검색 증강 생성)**: 로컬 벡터 DB와 한국어 임베딩을 통한 교안/문서 검색 및 출처(Citations) 표시
> 4. **보안 강화 및 Swagger 자물쇠**: `APIKeyHeader`를 통한 `/docs` 인터랙티브 인증 지원
> 5. **프리미엄 Streamlit UI/UX**: 글래스모피즘 디자인, 퀵 프롬프트 칩, 대화 내보내기, 실시간 지연시간 통계

In [1]:
# 서버 실행 도우미 — 노트북 맨 처음에 한 번 실행하세요.
import os, sys, asyncio, threading, time, socket, contextlib, importlib
import uvicorn

if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

for _d in ('app', 'models', 'data', 'frontend', 'data/chroma'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다.')
        return None
        
    # v2 개선: app 관련 서브모듈 캐시 정리
    if isinstance(app, str):
        mod_prefix = app.split(':')[0].split('.')[0]
        to_delete = [k for k in sys.modules.keys() if k.startswith(mod_prefix)]
        for k in to_delete:
            sys.modules.pop(k, None)

    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
        
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    
    def _run():
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
        
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    
    for i in range(600):
        if _port_open(host, port):
            print(f'✅ 서버 실행됨: http://{host}:{port}')
            return server
        if not thread.is_alive():
            print('❌ 서버 스레드가 종료됐습니다.')
            return server
        if i > 0 and i % 20 == 0:
            print(f'  ... 모델 및 벡터 DB 로드 중 ({i//2}초 경과)')
        time.sleep(0.5)
        
    print('5분 내에 서버가 시작되지 않았습니다.')
    return server

print('✅ 서버 도우미 준비 완료 (v2 RAG)')

✅ 서버 도우미 준비 완료 (v2 RAG)


In [2]:
# RAG 및 LLM 필수 패키지 점검
import importlib.util, subprocess, sys

for _pkg in ("transformers", "accelerate", "chromadb", "sentence_transformers"):
    if importlib.util.find_spec(_pkg) is None:
        print(f"📦 {_pkg} 설치 중...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

import torch, chromadb, sentence_transformers
print(f"PyTorch: {torch.__version__}")
print(f"ChromaDB: {chromadb.__version__}")
print(f"SentenceTransformers: {sentence_transformers.__version__}")

C:\Users\cheon\OneDrive\work\model-serving-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.13.0+cpu
ChromaDB: 1.5.9
SentenceTransformers: 6.0.0


## 2. 백엔드 v2 및 RAG 모듈 생성

In [3]:
%%writefile app/auth_v2.py
"""
Day 7 v2 - API Key 인증 모듈 (Swagger UI Authorize 연동)
"""
from fastapi import HTTPException, Security, status
from fastapi.security import APIKeyHeader

# API Key 헤더 스키마 정의 (Swagger UI 상단에 'Authorize' 자물쇠 버튼 활성화)
API_KEY_HEADER = APIKeyHeader(name="X-API-Key", auto_error=False)

# 인증에 사용할 유효한 API Key 목록 (실무에서는 환경 변수나 Secret Manager에서 로드)
VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
    "admin-key-999": "관리자",
}


async def verify_api_key(api_key: str = Security(API_KEY_HEADER)) -> str:
    """
    API Key를 검증하고 사용자 이름을 반환합니다.

    Args:
        api_key: HTTP 요청 헤더 'X-API-Key' 값

    Returns:
        str: 인증된 사용자 이름 (예: '사용자A')

    Raises:
        HTTPException(401): API Key가 누락되었거나 유효하지 않은 경우
    """
    if not api_key:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
            headers={"WWW-Authenticate": "ApiKey"},
        )

    if api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="유효하지 않은 API Key입니다.",
            headers={"WWW-Authenticate": "ApiKey"},
        )

    return VALID_API_KEYS[api_key]





Overwriting app/auth_v2.py


In [4]:
%%writefile app/chatbot_schemas_v2.py
"""
Day 7 v2 - 챗봇 Pydantic 입출력 데이터 스키마 (RAG 옵션 및 Citation 지원)
"""
from typing import Literal, Optional, List, Dict, Any
from pydantic import BaseModel, Field


class Message(BaseModel):
    """단일 대화 메시지 스키마"""
    role: Literal["user", "assistant", "bot", "system"] = Field(
        ...,
        description="발화자 역할 ('user', 'assistant', 'bot', 'system')",
        examples=["user"]
    )
    content: str = Field(
        ...,
        min_length=1,
        max_length=4000,
        description="메시지 텍스트 본문 (1자 ~ 4,000자)",
        examples=["안녕하세요! 인공지능에 대해 알려주세요."]
    )


class CitationItem(BaseModel):
    """RAG 검색을 통해 인용된 문서 청크 메타데이터"""
    source: str = Field(description="참조 문서 파일명 또는 소스 식별자")
    similarity: float = Field(description="코사인 유사도 점수 (0.0 ~ 1.0)")
    content: str = Field(description="참조된 문서 발췌문")
    category: Optional[str] = Field(default="general", description="문서 분류")


class ChatRequest(BaseModel):
    """챗봇 추론 요청 스키마"""
    messages: List[Message] = Field(
        ...,
        min_length=1,
        max_length=50,
        description="멀티턴 대화 기록 목록 (최대 50개 턴)",
    )
    system_prompt: Optional[str] = Field(
        default="너는 친절하고 유능한 한국어 AI 챗봇이야. 반드시 한국어로 정중하고 자연스럽게 대답해줘.",
        max_length=1000,
        description="챗봇의 페르소나/역할을 정의하는 시스템 프롬프트"
    )
    max_new_tokens: int = Field(
        default=150,
        ge=10,
        le=1000,
        description="최대 생성 토큰 수 (10 ~ 1000)"
    )
    temperature: float = Field(
        default=0.7,
        ge=0.0,
        le=2.0,
        description="생성 다양성 조절 (0.0=결정론적, 2.0=매우 창의적)"
    )
    top_p: float = Field(
        default=0.9,
        gt=0.0,
        le=1.0,
        description="Nucleus Sampling 확률 임계값"
    )
    # ===== RAG 옵션 =====
    use_rag: bool = Field(
        default=True,
        description="ChromaDB 기반 RAG 지식 검색 활성화 여부"
    )
    top_k: int = Field(
        default=3,
        ge=1,
        le=10,
        description="RAG 검색 시 인용할 관련 문서 청크 수 (1~10)"
    )

    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "messages": [
                        {"role": "user", "content": "FastAPI에서 PyTorch 추론 시 주의할 점은?"}
                    ],
                    "system_prompt": "너는 친절한 AI 어시스턴트야.",
                    "max_new_tokens": 150,
                    "temperature": 0.7,
                    "top_p": 0.9,
                    "use_rag": True,
                    "top_k": 3,
                }
            ]
        }
    }


class ChatResponse(BaseModel):
    """챗봇 추론 응답 스키마"""
    success: bool = Field(default=True, description="성공 여부")
    response: str = Field(description="생성된 챗봇 응답 텍스트")
    model_name: str = Field(description="추론에 사용된 모델 식별자")
    user: Optional[str] = Field(default=None, description="인증된 사용자 ID")
    tokens_generated: Optional[int] = Field(default=None, description="생성된 토큰 수")
    citations: Optional[List[CitationItem]] = Field(default=None, description="RAG 참조 문서 목록")


class DocumentUploadResponse(BaseModel):
    """문서 업로드 및 인덱싱 응답"""
    success: bool = Field(description="업로드 및 인덱싱 성공 여부")
    filename: str = Field(description="저장된 파일명")
    chunks_indexed: int = Field(description="생성된 벡터 청크 수")
    message: str = Field(description="결과 메시지")



Overwriting app/chatbot_schemas_v2.py


In [5]:
%%writefile app/chatbot_model_v2.py
"""
Day 7 v2 - 한국어 챗봇 추론 엔진 (슬라이딩 윈도우 수정 및 실시간 스트리밍 지원)
"""
import threading
from typing import Generator, List, Dict, Any, Optional
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer


class ChatbotModelV2:
    """Hugging Face CausalLM 기반 인스트럭트 챗봇 모델 엔진 v2"""

    def __init__(self, model_name: Optional[str] = None):
        # 1. 최적 디바이스 선택
        if torch.cuda.is_available():
            self.device = "cuda"
        elif torch.backends.mps.is_available():
            self.device = "mps"
        else:
            self.device = "cpu"

        # 2. 모델 식별자 자동 선택 (GPU 유무에 따라 1.5B 또는 0.5B)
        if model_name is None:
            if self.device != "cpu":
                model_name = "Qwen/Qwen2.5-1.5B-Instruct"
            else:
                model_name = "Qwen/Qwen2.5-0.5B-Instruct"

        self.model_name = model_name

        # 3. 토크나이저 로드
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        # 4. 정밀도(dtype) 결정
        if self.device == "cuda" and torch.cuda.is_bf16_supported():
            dtype = torch.bfloat16
        elif self.device in ("cuda", "mps"):
            dtype = torch.float16
        else:
            dtype = torch.float32

        # 5. 모델 로드 및 평가 모드 설정
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            trust_remote_code=True,
        )
        self.model = self.model.to(self.device)
        self.model.eval()

    def _prepare_chat_input(
        self,
        messages: List[Dict[str, str]],
        system_prompt: str,
        max_new_tokens: int,
    ) -> torch.Tensor:
        """대화 기록을 ChatML 형식으로 변환하고 2턴 단위 슬라이딩 윈도우를 적용합니다."""
        chat: List[Dict[str, str]] = [{"role": "system", "content": system_prompt}]
        for msg in messages:
            role = msg.get("role", "user")
            if role in ("bot", "assistant"):
                norm_role = "assistant"
            elif role == "system":
                norm_role = "system"
            else:
                norm_role = "user"
            chat.append({"role": norm_role, "content": msg.get("content", "")})

        def encode_chat(chat_list: List[Dict[str, str]]) -> torch.Tensor:
            encoded = self.tokenizer.apply_chat_template(
                chat_list,
                add_generation_prompt=True,
                return_tensors="pt"
            )
            if not torch.is_tensor(encoded):
                encoded = encoded["input_ids"]
            return encoded.to(self.device)

        input_ids = encode_chat(chat)
        max_context_length = getattr(self.model.config, "max_position_embeddings", 2048)

        # ✅ 핵심 수정: 턴(Turn) 구조 보존을 위해 [User, Assistant] 2개 단위로 안전하게 삭제
        while input_ids.shape[1] > (max_context_length - max_new_tokens) and len(chat) > 3:
            chat.pop(1)  # 가장 오래된 User 발화 삭제
            if len(chat) > 1 and chat[1]["role"] == "assistant":
                chat.pop(1)  # 그에 대응하는 Assistant 응답 삭제
            input_ids = encode_chat(chat)

        return input_ids

    def generate_response(
        self,
        messages: List[Dict[str, str]],
        system_prompt: str = "너는 친절하고 유능한 한국어 AI 어시스턴트야.",
        max_new_tokens: int = 150,
        temperature: float = 0.7,
        top_p: float = 0.9,
    ) -> Dict[str, Any]:
        """단일 요청에 대한 전체 텍스트 응답을 생성합니다."""
        input_ids = self._prepare_chat_input(messages, system_prompt, max_new_tokens)
        input_length = input_ids.shape[1]

        do_sample = temperature > 0.0
        gen_kwargs: Dict[str, Any] = {
            "input_ids": input_ids,
            "max_new_tokens": max_new_tokens,
            "top_p": top_p,
            "do_sample": do_sample,
            "pad_token_id": self.tokenizer.pad_token_id,
            "eos_token_id": self.tokenizer.eos_token_id,
        }
        if do_sample:
            gen_kwargs["temperature"] = temperature

        with torch.inference_mode():  # no_grad보다 빠른 고속 추론 모드
            output_ids = self.model.generate(**gen_kwargs)

        new_tokens = output_ids[0][input_length:]
        response_text = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        tokens_generated = int(new_tokens.shape[0])

        return {
            "response": response_text if response_text else "(응답을 생성하지 못했습니다.)",
            "tokens_generated": tokens_generated,
        }

    def generate_stream(
        self,
        messages: List[Dict[str, str]],
        system_prompt: str = "너는 친절하고 유능한 한국어 AI 어시스턴트야.",
        max_new_tokens: int = 150,
        temperature: float = 0.7,
        top_p: float = 0.9,
    ) -> Generator[str, None, None]:
        """실시간 토큰 스트리밍 출력을 제너레이터 형태로 반환합니다."""
        input_ids = self._prepare_chat_input(messages, system_prompt, max_new_tokens)

        # 토큰 스트리머 생성 (특수 토큰 자동 스킵)
        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        do_sample = temperature > 0.0
        gen_kwargs: Dict[str, Any] = {
            "input_ids": input_ids,
            "max_new_tokens": max_new_tokens,
            "top_p": top_p,
            "do_sample": do_sample,
            "pad_token_id": self.tokenizer.pad_token_id,
            "eos_token_id": self.tokenizer.eos_token_id,
            "streamer": streamer,
        }
        if do_sample:
            gen_kwargs["temperature"] = temperature

        # 백그라운드 스레드에서 모델 생성 실행
        thread = threading.Thread(target=self._run_generation, kwargs={"gen_kwargs": gen_kwargs})
        thread.start()

        # 실시간으로 생성되는 텍스트 청크를 yield
        for text_chunk in streamer:
            yield text_chunk

        thread.join()

    def _run_generation(self, gen_kwargs: Dict[str, Any]) -> None:
        with torch.inference_mode():
            self.model.generate(**gen_kwargs)





Overwriting app/chatbot_model_v2.py


In [6]:
%%writefile app/rag_engine.py
"""
Day 7 v2 RAG - ChromaDB 및 SentenceTransformers 기반 검색 증강 생성(RAG) 엔진
"""
import os
import glob
from typing import List, Dict, Any, Optional
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer


class RAGEngine:
    """ChromaDB 로컬 벡터 데이터베이스 기반 경량 RAG 엔진"""

    def __init__(
        self,
        persist_dir: str = "data/chroma",
        embedding_model_name: str = "jhgan/ko-sroberta-multitask",
        collection_name: str = "aiffel_course_kb"
    ):
        self.persist_dir = persist_dir
        os.makedirs(self.persist_dir, exist_ok=True)

        # 1. 한국어 최적화 임베딩 모델 로드 (가볍고 고성능)
        print(f"📦 [RAG] 임베딩 모델 로드 중: {embedding_model_name}")
        self.embedder = SentenceTransformer(embedding_model_name)
        print("✅ [RAG] 임베딩 모델 로드 완료")

        # 2. ChromaDB 영구 저장소 클라이언트 초기화
        self.client = chromadb.PersistentClient(path=self.persist_dir)
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

        # 3. 기본 지식베이스 자동 인덱싱 (README.md, quests/*.md)
        self.auto_index_default_docs()

    def chunk_text(self, text: str, chunk_size: int = 400, overlap: int = 60) -> List[str]:
        """텍스트를 문맥 보존 슬라이딩 윈도우 방식으로 청킹합니다."""
        if not text:
            return []
        
        # 줄바꿈 기준 1차 분할
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        chunks = []
        current_chunk = ""

        for p in paragraphs:
            if len(current_chunk) + len(p) <= chunk_size:
                current_chunk += ("\n\n" + p if current_chunk else p)
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                # 단일 문단이 chunk_size보다 큰 경우 글자 단위 분할
                if len(p) > chunk_size:
                    for i in range(0, len(p), chunk_size - overlap):
                        sub_chunk = p[i:i + chunk_size]
                        if sub_chunk:
                            chunks.append(sub_chunk)
                    current_chunk = ""
                else:
                    current_chunk = p

        if current_chunk:
            chunks.append(current_chunk)

        return chunks

    def add_document(self, content: str, source_name: str, doc_category: str = "manual") -> int:
        """단일 문서 내용을 청킹하여 벡터 DB에 인덱싱합니다."""
        chunks = self.chunk_text(content)
        if not chunks:
            return 0

        # 중복 방지를 위한 ID 생성
        safe_source = os.path.basename(source_name).replace(" ", "_")
        ids = [f"{safe_source}_chunk_{i}" for i in range(len(chunks))]
        metadatas = [
            {"source": safe_source, "category": doc_category, "chunk_index": i}
            for i in range(len(chunks))
        ]

        # 텍스트 임베딩 계산
        embeddings = self.embedder.encode(chunks, show_progress_bar=False).tolist()

        # ChromaDB에 upsert
        self.collection.upsert(
            ids=ids,
            embeddings=embeddings,
            documents=chunks,
            metadatas=metadatas
        )
        return len(chunks)

    def retrieve(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """사용자 질문과 코사인 유사도가 가장 높은 상위 K개 문서 청크 검색"""
        if self.collection.count() == 0:
            return []

        query_emb = self.embedder.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_emb,
            n_results=min(top_k, self.collection.count()),
            include=["documents", "metadatas", "distances"]
        )

        retrieved_items = []
        if results and results.get("documents") and results["documents"][0]:
            docs = results["documents"][0]
            metas = results["metadatas"][0]
            dists = results["distances"][0]

            for doc, meta, dist in zip(docs, metas, dists):
                # Cosine 거리(0~2) -> 유사도 점수(0~1) 변환
                similarity = round(max(0.0, 1.0 - float(dist)), 3)
                retrieved_items.append({
                    "content": doc,
                    "source": meta.get("source", "알 수 없음"),
                    "category": meta.get("category", "general"),
                    "chunk_index": meta.get("chunk_index", 0),
                    "similarity": similarity,
                })

        return retrieved_items

    def auto_index_default_docs(self) -> None:
        """저장소 내의 핵심 코스 문서들을 확인하고 자동 인덱싱합니다."""
        existing_count = self.collection.count()
        if existing_count > 0:
            return

        print("📚 [RAG] 기본 코스 문서 인덱싱을 시작합니다...")
        doc_files = []
        
        # 1. README.md
        if os.path.exists("README.md"):
            doc_files.append(("README.md", "course_overview"))

        # 2. quests/*.md
        for qf in glob.glob("quests/Quest*.md"):
            doc_files.append((qf, "quest_report"))

        total_indexed = 0
        for fpath, cat in doc_files:
            try:
                with open(fpath, "r", encoding="utf-8") as f:
                    content = f.read()
                count = self.add_document(content, source_name=fpath, doc_category=cat)
                total_indexed += count
            except Exception as e:
                print(f"⚠️ {fpath} 인덱싱 실패: {e}")

        print(f"✅ [RAG] 기본 문서 {len(doc_files)}개 파일 ({total_indexed}개 청크) 색인 완료")

    def get_stats(self) -> Dict[str, Any]:
        """현재 인덱싱된 지식 데이터베이스 통계 조회"""
        count = self.collection.count()
        all_meta = self.collection.get(include=["metadatas"])
        sources = set()
        if all_meta and all_meta.get("metadatas"):
            for m in all_meta["metadatas"]:
                if m and "source" in m:
                    sources.add(m["source"])

        return {
            "total_chunks": count,
            "unique_documents": list(sources),
            "doc_count": len(sources),
            "storage_path": self.persist_dir,
        }



Overwriting app/rag_engine.py


In [7]:
%%writefile app/chatbot_api_v2.py
"""
Day 7 v2 - 한국어 챗봇 FastAPI 서버 (ChromaDB RAG 검색 증강 생성 및 실시간 스트리밍 지원)
"""
import asyncio
import json
import urllib.parse
from contextlib import asynccontextmanager
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List, Dict, Any

from fastapi import FastAPI, Depends, HTTPException, UploadFile, File, Form, status
from fastapi.responses import StreamingResponse

from app.chatbot_schemas_v2 import ChatRequest, ChatResponse, DocumentUploadResponse, CitationItem
from app.chatbot_model_v2 import ChatbotModelV2
from app.auth_v2 import verify_api_key
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware
from app.rag_engine import RAGEngine


# ===== 설정 & 로거 =====
logger = setup_logger("chatbot_api_v2")
inference_executor = ThreadPoolExecutor(max_workers=2, thread_name_prefix="chatbot_v2")

# ===== 전역 모델 & RAG 엔진 인스턴스 =====
chatbot_engine: Optional[ChatbotModelV2] = None
rag_engine: Optional[RAGEngine] = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    """서버 시작 시 챗봇 모델 및 RAG 벡터 DB 엔진을 사전 로드합니다."""
    global chatbot_engine, rag_engine
    logger.info("🚀 [v2] 챗봇 및 RAG 엔진 초기화 시작...")
    try:
        # 1. LLM 추론 엔진 로드
        chatbot_engine = ChatbotModelV2()
        logger.info(f"✅ [LLM] 모델 로드 완료: {chatbot_engine.model_name} (디바이스: {chatbot_engine.device})")

        # 2. ChromaDB RAG 엔진 로드
        rag_engine = RAGEngine()
        logger.info("✅ [RAG] ChromaDB 벡터 데이터베이스 로드 및 기본 문서 색인 완료")
    except Exception as e:
        logger.error(f"❌ 엔진 로드 실패: {e}")
        raise e

    yield

    logger.info("🛑 [v2] 서버 종료 및 자원 정리")
    inference_executor.shutdown(wait=False)


# ===== FastAPI 앱 생성 =====
app = FastAPI(
    title="Korean LLM Chatbot with RAG API v2",
    description="Hugging Face Qwen2.5 & ChromaDB RAG 기반 지식 검색 챗봇 서비스 (스트리밍 & API Key 인증)",
    version="2.1.0",
    lifespan=lifespan,
)

# 미들웨어 및 에러 핸들러 등록
app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)


# ===== RAG 프롬프트 합성 헬퍼 =====
def _compose_rag_prompt(user_query: str, base_system_prompt: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    """검색된 문서 청크들을 시스템 프롬프트에 주입하여 지식 기반 프롬프트를 구성합니다."""
    if not retrieved_chunks:
        return base_system_prompt

    context_parts = []
    for idx, chunk in enumerate(retrieved_chunks):
        source = chunk.get("source", "문서")
        similarity = chunk.get("similarity", 0.0)
        content = chunk.get("content", "").strip()
        context_parts.append(f"[참고자료 {idx + 1}] (출처: {source} / 신뢰도: {int(similarity * 100)}%)\n{content}")

    context_text = "\n\n".join(context_parts)

    rag_prompt = (
        f"{base_system_prompt}\n\n"
        f"반드시 아래 제공된 [참고자료]의 사실에 기반하여 사용자의 질문에 정확하고 구체적으로 답변해줘.\n"
        f"만약 참고자료에서 질문에 대한 답을 찾을 수 없다면 억지로 지어내지 말고 솔직하게 '제공된 문서에서 확인할 수 없습니다'라고 답변해.\n\n"
        f"=== [참고자료 시작] ===\n"
        f"{context_text}\n"
        f"=== [참고자료 끝] ==="
    )
    return rag_prompt


def _run_batch_chat(req_dict: dict) -> dict:
    """별도 스레드에서 실행되는 배치 추론 함수"""
    if chatbot_engine is None:
        raise RuntimeError("챗봇 모델 엔진이 준비되지 않았습니다.")
    return chatbot_engine.generate_response(
        messages=req_dict["messages"],
        system_prompt=req_dict["system_prompt"],
        max_new_tokens=req_dict["max_new_tokens"],
        temperature=req_dict["temperature"],
        top_p=req_dict["top_p"],
    )


# ===== 엔드포인트 정의 =====

@app.get("/health", tags=["System"])
async def health_check():
    """서버, 모델 엔진 및 RAG 벡터 DB 상태 확인"""
    is_ready = (chatbot_engine is not None) and (rag_engine is not None)
    rag_stats = rag_engine.get_stats() if rag_engine else {}
    return {
        "status": "healthy" if is_ready else "loading",
        "model_name": chatbot_engine.model_name if chatbot_engine else None,
        "device": chatbot_engine.device if chatbot_engine else None,
        "rag_ready": rag_engine is not None,
        "rag_total_chunks": rag_stats.get("total_chunks", 0),
        "version": "2.1.0",
    }


@app.get("/rag/stats", tags=["RAG"])
async def get_rag_stats(user: str = Depends(verify_api_key)):
    """현재 인덱싱된 RAG 지식 데이터베이스 통계 조회"""
    if rag_engine is None:
        raise HTTPException(status_code=503, detail="RAG 엔진이 초기화되지 않았습니다.")
    return {
        "success": True,
        "user": user,
        "stats": rag_engine.get_stats(),
    }


@app.post("/rag/upload", response_model=DocumentUploadResponse, tags=["RAG"])
async def upload_document_for_rag(
    file: UploadFile = File(...),
    category: str = Form("custom_upload"),
    user: str = Depends(verify_api_key),
):
    """
    [문서 업로드] 텍스트(.txt, .md) 문서를 업로드 받아 즉시 청킹 및 벡터 DB에 인덱싱합니다.
    """
    if rag_engine is None:
        raise HTTPException(status_code=503, detail="RAG 엔진이 초기화되지 않았습니다.")

    filename = file.filename or "unknown_file.txt"
    if not (filename.endswith(".txt") or filename.endswith(".md") or filename.endswith(".py") or filename.endswith(".json")):
        raise HTTPException(
            status_code=400,
            detail="지원되지 않는 파일 형식입니다. (.txt, .md, .py, .json 파일만 지원됩니다)"
        )

    try:
        content_bytes = await file.read()
        content = content_bytes.decode("utf-8", errors="replace")
        chunks_count = rag_engine.add_document(
            content=content,
            source_name=filename,
            doc_category=category
        )
        logger.info(f"📚 [RAG 업로드] 사용자: {user} | 파일: {filename} ({chunks_count}개 청크 인덱싱)")
        return DocumentUploadResponse(
            success=True,
            filename=filename,
            chunks_indexed=chunks_count,
            message=f"성공적으로 {chunks_count}개의 지식 청크가 벡터 DB에 추가되었습니다."
        )
    except Exception as e:
        logger.error(f"❌ 문서 인덱싱 실패: {e}")
        raise HTTPException(status_code=500, detail=f"문서 인덱싱 오류: {str(e)}")


@app.post("/chat", response_model=ChatResponse, tags=["Chat"])
async def chat_batch(
    request: ChatRequest,
    user: str = Depends(verify_api_key),
):
    """
    [일반 추론] RAG 지식 검색(선택) 후 전체 텍스트 응답 및 출처(Citations)를 반환합니다.
    """
    if chatbot_engine is None or rag_engine is None:
        raise HTTPException(status_code=503, detail="서버 엔진이 아직 준비 중입니다.")

    logger.info(f"📩 [일반추론] 사용자: {user} | RAG: {request.use_rag} | 메시지 수: {len(request.messages)}")

    # 1. RAG 지식 검색 수행 (활성화된 경우)
    citations: Optional[List[CitationItem]] = None
    final_system_prompt = request.system_prompt or "너는 친절한 AI 어시스턴트야."

    if request.use_rag and request.messages:
        last_user_query = next((m.content for m in reversed(request.messages) if m.role == "user"), "")
        if last_user_query:
            raw_retrieved = rag_engine.retrieve(query=last_user_query, top_k=request.top_k)
            if raw_retrieved:
                final_system_prompt = _compose_rag_prompt(last_user_query, final_system_prompt, raw_retrieved)
                citations = [
                    CitationItem(
                        source=c["source"],
                        similarity=c["similarity"],
                        content=c["content"],
                        category=c["category"],
                    )
                    for c in raw_retrieved
                ]

    try:
        loop = asyncio.get_running_loop()
        req_data = {
            "messages": [m.model_dump() for m in request.messages],
            "system_prompt": final_system_prompt,
            "max_new_tokens": request.max_new_tokens,
            "temperature": request.temperature,
            "top_p": request.top_p,
        }

        result = await loop.run_in_executor(inference_executor, _run_batch_chat, req_data)
    except Exception as e:
        logger.error(f"❌ 추론 실패: {e}")
        raise HTTPException(status_code=500, detail=f"응답 생성 실패: {str(e)}")

    return ChatResponse(
        success=True,
        response=result["response"],
        model_name=chatbot_engine.model_name,
        user=user,
        tokens_generated=result["tokens_generated"],
        citations=citations,
    )


@app.post("/chat/stream", tags=["Chat"])
async def chat_stream(
    request: ChatRequest,
    user: str = Depends(verify_api_key),
):
    """
    [실시간 스트리밍] RAG 지식 검색(선택) 후 실시간 토큰 스트림 및 HTTP 헤더 출처를 전송합니다.
    """
    if chatbot_engine is None or rag_engine is None:
        raise HTTPException(status_code=503, detail="서버 엔진이 아직 준비 중입니다.")

    logger.info(f"🌊 [스트리밍] 사용자: {user} | RAG: {request.use_rag} | 메시지 수: {len(request.messages)}")

    # 1. RAG 지식 검색 수행
    citations_data = []
    final_system_prompt = request.system_prompt or "너는 친절한 AI 어시스턴트야."

    if request.use_rag and request.messages:
        last_user_query = next((m.content for m in reversed(request.messages) if m.role == "user"), "")
        if last_user_query:
            raw_retrieved = rag_engine.retrieve(query=last_user_query, top_k=request.top_k)
            if raw_retrieved:
                final_system_prompt = _compose_rag_prompt(last_user_query, final_system_prompt, raw_retrieved)
                citations_data = raw_retrieved

    # 2. 스트리밍 제너레이터 함수
    def stream_generator():
        try:
            req_messages = [m.model_dump() for m in request.messages]
            generator = chatbot_engine.generate_stream(
                messages=req_messages,
                system_prompt=final_system_prompt,
                max_new_tokens=request.max_new_tokens,
                temperature=request.temperature,
                top_p=request.top_p,
            )
            for chunk in generator:
                yield chunk
        except Exception as e:
            logger.error(f"❌ 스트리밍 오류: {e}")
            yield f"\n[생성 오류: {str(e)}]"

    # 출처 메타데이터를 URL-Encoded JSON 헤더로 전달 (안전한 아스키 문자열)
    encoded_citations = urllib.parse.quote(json.dumps(citations_data, ensure_ascii=False))

    return StreamingResponse(
        stream_generator(),
        media_type="text/plain; charset=utf-8",
        headers={
            "X-Model-Name": chatbot_engine.model_name,
            "X-User": user,
            "X-Citations": encoded_citations,
            "Access-Control-Expose-Headers": "X-Citations, X-Model-Name, X-User",
            "Cache-Control": "no-cache",
        }
    )



Overwriting app/chatbot_api_v2.py


## 3. 프론트엔드 v2 대시보드 생성 (Streamlit RAG)

In [8]:
%%writefile frontend/app_chatbot_v2.py
"""
Day 7 v2 RAG Enhanced - 프리미엄 글래스모피즘 한국어 LLM 챗봇 대시보드 (RAG 지식 검색 & 문서 업로드 & 출처 표시)
"""
import streamlit as st
import requests
import json
import urllib.parse
import time
from datetime import datetime
from typing import Generator, List, Dict, Any, Tuple

# ===== 1. 페이지 환경 설정 =====
st.set_page_config(
    page_title="AIFFEL RAG Chatbot v2",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded",
)

API_BASE = "http://localhost:8000"

# ===== 2. 모던 글래스모피즘 커스텀 CSS 주입 =====
CUSTOM_CSS = """
<style>
@import url('https://cdn.jsdelivr.net/gh/orioncactus/pretendard/dist/web/static/pretendard.css');

html, body, [class*="css"] {
    font-family: 'Pretendard', -apple-system, BlinkMacSystemFont, system-ui, Roboto, sans-serif;
}

.stApp {
    background: radial-gradient(circle at 10% 20%, rgba(15, 23, 42, 0.95) 0%, rgba(2, 6, 23, 1) 90%);
}

/* 글래스모피즘 카드 */
.glass-card {
    background: rgba(30, 41, 59, 0.45);
    backdrop-filter: blur(16px);
    -webkit-backdrop-filter: blur(16px);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 16px;
    padding: 20px;
    box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.37);
    margin-bottom: 16px;
    transition: all 0.3s ease;
}

.gradient-title {
    background: linear-gradient(135deg, #60A5FA 0%, #A78BFA 50%, #F472B6 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.2rem;
    font-weight: 800;
    letter-spacing: -0.02em;
    margin-bottom: 4px;
}

.subtitle-text {
    color: #94A3B8;
    font-size: 0.95rem;
    margin-bottom: 20px;
}

/* 뱃지 스타일 */
.status-badge {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    padding: 4px 12px;
    border-radius: 9999px;
    font-size: 0.8rem;
    font-weight: 600;
}

.badge-online {
    background: rgba(34, 197, 94, 0.15);
    color: #4ADE80;
    border: 1px solid rgba(34, 197, 94, 0.3);
}

.badge-offline {
    background: rgba(239, 68, 68, 0.15);
    color: #F87171;
    border: 1px solid rgba(239, 68, 68, 0.3);
}

.badge-rag {
    background: rgba(234, 179, 8, 0.15);
    color: #FACC15;
    border: 1px solid rgba(234, 179, 8, 0.3);
}

.badge-model {
    background: rgba(99, 102, 241, 0.15);
    color: #818CF8;
    border: 1px solid rgba(99, 102, 241, 0.3);
}

/* 출처 인용 카드 */
.citation-card {
    background: rgba(15, 23, 42, 0.6);
    border-left: 3px solid #6366F1;
    border-radius: 6px;
    padding: 10px 14px;
    margin-bottom: 8px;
    font-size: 0.85rem;
}

.citation-header {
    display: flex;
    justify-content: space-between;
    font-weight: 600;
    color: #CBD5E1;
    margin-bottom: 4px;
}

/* 퀵 프롬프트 카드 */
.starter-card {
    background: rgba(30, 41, 59, 0.3);
    border: 1px solid rgba(255, 255, 255, 0.06);
    border-radius: 12px;
    padding: 14px;
    transition: all 0.2s ease;
}

.starter-card:hover {
    background: rgba(99, 102, 241, 0.15);
    border-color: rgba(99, 102, 241, 0.4);
    transform: translateY(-2px);
}

[data-testid="stChatMessage"] {
    background: rgba(30, 41, 59, 0.4) !important;
    border: 1px solid rgba(255, 255, 255, 0.06) !important;
    border-radius: 16px !important;
    padding: 16px !important;
    margin-bottom: 12px !important;
}

section[data-testid="stSidebar"] {
    background: rgba(15, 23, 42, 0.8) !important;
    border-right: 1px solid rgba(255, 255, 255, 0.08) !important;
}
</style>
"""
st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

# ===== 3. 페르소나 및 퀵 프롬프트 프리셋 =====
PERSONA_PRESETS = {
    "🌟 친절한 AI 어시스턴트": {
        "prompt": "너는 다정하고 친절한 한국어 AI 어시스턴트야. 사용자의 질문에 항상 경청하며 명확하고 상냥하게 대답해줘.",
        "avatar": "🤖",
        "desc": "일반적인 질의응답 및 친절한 대화 지원"
    },
    "💻 시니어 ML/배포 멘토": {
        "prompt": "너는 15년 차 시니어 머신러닝 배포 엔지니어 멘토야. PyTorch 모델 서빙, FastAPI, TorchScript, ONNX, 비동기 최적화에 대해 실무 관점의 모범 답안을 제공해줘.",
        "avatar": "🧑‍💻",
        "desc": "AIFFEL 모델 서빙 교안 및 코드 전문 멘토"
    },
    "🌐 전문 번역/글쓰기 에디터": {
        "prompt": "너는 한영 전문 번역가이자 테크니컬 라이터야. 문맥에 알맞은 자연스러운 번역과 문서 교정을 전문으로 해.",
        "avatar": "✍️",
        "desc": "자연스러운 한/영 번역 및 문서 교정"
    },
    "⚡ 비즈니스 핵심 요약 봇": {
        "prompt": "너는 핵심만 짚어주는 요약 전문가야. 군더더기 없이 불렛포인트(•) 3~5줄 이내로 핵심만 명확히 요약해.",
        "avatar": "⚡",
        "desc": "빠르고 간결한 3줄 요약"
    },
    "✏️ 직접 설정 (Custom)": {
        "prompt": "",
        "avatar": "🎯",
        "desc": "사용자 정의 시스템 프롬프트"
    },
}

QUICK_STARTERS = [
    {"title": "📚 RAG 지식: run_in_executor 역할", "prompt": "FastAPI에서 PyTorch 추론 함수를 실행할 때 run_in_executor와 ThreadPoolExecutor를 사용하는 이유를 코스 문서를 기반으로 설명해줘.", "icon": "⚡"},
    {"title": "📦 RAG 지식: state_dict vs TorchScript", "prompt": "Day 1 교안에서 배운 state_dict, TorchScript, ONNX 포맷의 장단점과 배포 차이를 비교해줘.", "icon": "🧠"},
    {"title": "🔒 RAG 지식: API Key 보안 안전장치", "prompt": "Day 6 미디어 서빙에서 API Key 인증 헤더와 파일 크기/MIME 타입 검증의 구현 방식을 알려줘.", "icon": "🛡️"},
    {"title": "📝 RAG 지식: 전체 서빙 코스 요약", "prompt": "이 저장소의 README를 바탕으로 4일간 진행된 모델 서빙 프로젝트의 전체 파이프라인 구조를 요약해줘.", "icon": "📋"},
]


# ===== 4. 세션 상태 초기화 =====
if "chat_history" not in st.session_state:
    st.session_state["chat_history"] = []

if "latency_history" not in st.session_state:
    st.session_state["latency_history"] = []

if "total_tokens_generated" not in st.session_state:
    st.session_state["total_tokens_generated"] = 0

if "pending_input" not in st.session_state:
    st.session_state["pending_input"] = None

if "last_citations" not in st.session_state:
    st.session_state["last_citations"] = []


# ===== 5. API 헬퍼 함수들 =====
@st.cache_data(ttl=3, show_spinner=False)
def get_server_health(base_url: str) -> Dict[str, Any]:
    try:
        resp = requests.get(f"{base_url}/health", timeout=2)
        if resp.status_code == 200:
            return resp.json()
    except Exception:
        pass
    return {"status": "offline"}


def get_rag_stats(api_key: str) -> Dict[str, Any]:
    try:
        resp = requests.get(f"{API_BASE}/rag/stats", headers={"X-API-Key": api_key}, timeout=2)
        if resp.status_code == 200:
            return resp.json().get("stats", {})
    except Exception:
        pass
    return {}


def upload_rag_file(uploaded_file, api_key: str) -> Tuple[bool, str]:
    try:
        files = {"file": (uploaded_file.name, uploaded_file.getvalue(), uploaded_file.type)}
        resp = requests.post(f"{API_BASE}/rag/upload", files=files, headers={"X-API-Key": api_key}, timeout=30)
        if resp.status_code == 200:
            res = resp.json()
            return True, f"✅ '{res['filename']}' ({res['chunks_indexed']}개 청크 색인 완료)"
        else:
            return False, f"❌ 업로드 실패: {resp.text}"
    except Exception as e:
        return False, f"❌ 업로드 오류: {str(e)}"


def stream_chat_api_rag(
    messages: List[Dict[str, str]],
    system_prompt: str,
    api_key: str,
    max_tokens: int,
    temperature: float,
    top_p: float,
    use_rag: bool,
    top_k: int,
) -> Tuple[Generator[str, None, None], List[Dict[str, Any]]]:
    """RAG 활성화 스트리밍 제너레이터 및 출처 메타데이터 추출"""
    payload = {
        "messages": messages,
        "system_prompt": system_prompt,
        "max_new_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "use_rag": use_rag,
        "top_k": top_k,
    }
    headers = {"X-API-Key": api_key}
    citations = []

    try:
        response = requests.post(
            f"{API_BASE}/chat/stream",
            json=payload,
            headers=headers,
            stream=True,
            timeout=120,
        )

        if response.status_code == 401:
            def err_gen():
                yield "❌ **[인증 실패]** API Key가 유효하지 않습니다."
            return err_gen(), []
        elif response.status_code != 200:
            def err_gen():
                yield f"❌ **[서버 오류]** HTTP {response.status_code}: {response.text}"
            return err_gen(), []

        # 헤더에서 RAG 출처 추출 (URL Encoded JSON)
        raw_citations_header = response.headers.get("X-Citations")
        if raw_citations_header:
            try:
                decoded_str = urllib.parse.unquote(raw_citations_header)
                citations = json.loads(decoded_str)
            except Exception:
                citations = []

        def token_generator():
            for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
                if chunk:
                    yield chunk

        return token_generator(), citations

    except requests.exceptions.ConnectionError:
        def err_gen():
            yield "🔌 **[연결 실패]** FastAPI 서버(포트 8000)와 통신할 수 없습니다."
        return err_gen(), []
    except Exception as e:
        def err_gen():
            yield f"❌ **[오류 발생]** {str(e)}"
        return err_gen(), []


def export_conversation_markdown(history: List[Dict[str, Any]], system_prompt: str) -> str:
    lines = [
        "# 💬 AIFFEL RAG 챗봇 대화 기록",
        f"- 생성 일시: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"- 시스템 페르소나: `{system_prompt}`",
        f"- 총 대화 턴: {len([m for m in history if m['role'] == 'user'])}턴",
        "\n---\n"
    ]
    for msg in history:
        role_name = "🧑 사용자 (User)" if msg["role"] == "user" else "🤖 챗봇 (AI)"
        timestamp = msg.get("timestamp", "")
        time_info = f" *({timestamp})*" if timestamp else ""
        lines.append(f"### {role_name}{time_info}\n")
        lines.append(f"{msg['content']}\n")
        if msg.get("citations"):
            lines.append("\n**📌 인용된 지식 출처:**\n")
            for c in msg["citations"]:
                lines.append(f"- 📄 `{c.get('source')}` (유사도: {int(c.get('similarity', 0)*100)}%)")
        if msg.get("latency"):
            lines.append(f"\n> ⏱️ 소요 시간: {msg['latency']}s | 생성 토큰: ~{msg.get('tokens', 0)}개\n")
        lines.append("\n---\n")
    return "\n".join(lines)


# ===== 6. 사이드바 레이아웃 =====
with st.sidebar:
    st.markdown("### ⚙️ 서비스 환경설정")

    api_key = st.text_input(
        "🔑 API Key",
        value="test-key-001",
        type="password",
        help="FastAPI 보안 인증 키 ('test-key-001' / 'test-key-002')"
    )

    st.markdown("---")
    st.markdown("### 🧠 RAG 지식 검색 설정")
    use_rag = st.toggle("📚 RAG 지식 검색 활성화", value=True, help="ChromaDB 벡터 데이터베이스에서 관련 문서를 검색하여 정확한 출처와 함께 답변합니다.")
    top_k = st.slider("인용할 문서 수 (Top-K)", 1, 5, 3, disabled=not use_rag)

    # 지식 파일 업로드 위젯
    with st.expander("📂 새 지식 문서 업로드"):
        uploaded = st.file_uploader("TXT, MD, PY 파일", type=["txt", "md", "py", "json"])
        if uploaded and st.button("벡터 DB에 색인하기", use_container_width=True):
            with st.spinner("임베딩 및 색인 중..."):
                ok, msg = upload_rag_file(uploaded, api_key)
                if ok:
                    st.success(msg)
                else:
                    st.error(msg)

    # RAG 통계 표시
    rag_info = get_rag_stats(api_key)
    if rag_info:
        st.caption(f"📊 총 색인 청크: **{rag_info.get('total_chunks', 0)}개** ({rag_info.get('doc_count', 0)}개 파일)")

    st.markdown("---")
    st.markdown("### 🎭 페르소나 설정")
    preset_choice = st.selectbox("프리셋 선택", list(PERSONA_PRESETS.keys()), index=1)
    preset_data = PERSONA_PRESETS[preset_choice]

    if preset_choice == "✏️ 직접 설정 (Custom)":
        active_system_prompt = st.text_area("시스템 프롬프트 직접 작성", value="너는 친절한 AI 어시스턴트야.", height=90)
        current_avatar = "🎯"
    else:
        active_system_prompt = preset_data["prompt"]
        current_avatar = preset_data["avatar"]
        st.caption(f"ℹ️ {preset_data['desc']}")

    st.markdown("---")
    st.markdown("### 🎛️ 생성 파라미터")
    max_tokens = st.slider("최대 토큰 (max_new_tokens)", 30, 600, 180, step=10)
    temperature = st.slider("Temperature (창의성)", 0.0, 1.5, 0.7, step=0.05)
    top_p = st.slider("Top-p", 0.1, 1.0, 0.9, step=0.05)

    st.markdown("---")
    st.markdown("### 📊 실시간 세션 통계")
    user_turns = len([m for m in st.session_state["chat_history"] if m["role"] == "user"])
    m_col1, m_col2 = st.columns(2)
    with m_col1:
        st.metric("대화 턴 수", f"{user_turns} 턴")
    with m_col2:
        st.metric("누적 토큰", f"{st.session_state['total_tokens_generated']} 개")

    if st.session_state["latency_history"]:
        avg_lat = round(sum(st.session_state["latency_history"]) / len(st.session_state["latency_history"]), 2)
        st.metric("평균 지연시간", f"{avg_lat} 초")
        st.line_chart(st.session_state["latency_history"][-10:], height=90)

    st.markdown("---")
    st.markdown("### 📥 대화 관리 & 다운로드")
    if st.session_state["chat_history"]:
        md_data = export_conversation_markdown(st.session_state["chat_history"], active_system_prompt)
        json_data = json.dumps(st.session_state["chat_history"], ensure_ascii=False, indent=2)

        d_col1, d_col2 = st.columns(2)
        with d_col1:
            st.download_button("📄 Markdown", data=md_data, file_name=f"rag_chat_{datetime.now().strftime('%m%d_%H%M')}.md", mime="text/markdown", use_container_width=True)
        with d_col2:
            st.download_button("📦 JSON", data=json_data, file_name=f"rag_chat_{datetime.now().strftime('%m%d_%H%M')}.json", mime="application/json", use_container_width=True)

    if st.button("🗑️ 대화 기록 초기화", use_container_width=True):
        st.session_state["chat_history"] = []
        st.session_state["latency_history"] = []
        st.session_state["total_tokens_generated"] = 0
        st.rerun()


# ===== 7. 메인 헤더 영역 =====
health = get_server_health(API_BASE)
is_online = health.get("status") == "healthy"
model_tag = health.get("model_name", "Qwen2.5-Instruct")
device_tag = health.get("device", "N/A").upper()
rag_status = health.get("rag_ready", False)

header_html = f"""
<div class="glass-card">
    <div style="display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 10px;">
        <div>
            <div class="gradient-title">🧠 AIFFEL Korean LLM & RAG Chatbot v2</div>
            <div class="subtitle-text">FastAPI 비동기 백엔드 & ChromaDB 로컬 벡터 지식베이스 실시간 스트리밍</div>
        </div>
        <div style="display: flex; gap: 8px; align-items: center;">
            <span class="status-badge {'badge-online' if is_online else 'badge-offline'}">
                {'🟢 서버 온라인' if is_online else '🔴 서버 오프라인'}
            </span>
            <span class="status-badge {'badge-rag' if rag_status else 'badge-offline'}">
                {'📚 RAG 준비됨' if rag_status else '⚠️ RAG 로딩중'}
            </span>
            <span class="status-badge badge-model">
                🧠 {model_tag} ({device_tag})
            </span>
        </div>
    </div>
</div>
"""
st.markdown(header_html, unsafe_allow_html=True)


# ===== 8. 대화가 비어 있을 때: 추천 RAG 질문 칩 표시 =====
if not st.session_state["chat_history"]:
    st.markdown("#### 💡 RAG 지식 기반 추천 질문으로 대화 시작하기")
    q_cols = st.columns(2)
    for idx, item in enumerate(QUICK_STARTERS):
        with q_cols[idx % 2]:
            st.markdown(
                f"""
                <div class="starter-card">
                    <div style="font-weight: 700; font-size: 0.95rem; color: #E2E8F0; margin-bottom: 4px;">
                        {item['icon']} {item['title']}
                    </div>
                    <div style="font-size: 0.85rem; color: #94A3B8;">
                        "{item['prompt']}"
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )
            if st.button(f"👉 질문하기: {item['title']}", key=f"quick_{idx}", use_container_width=True):
                st.session_state["pending_input"] = item["prompt"]
                st.rerun()

    st.markdown("<div style='height: 20px;'></div>", unsafe_allow_html=True)


# ===== 9. 대화 히스토리 출력 =====
for msg in st.session_state["chat_history"]:
    role = msg["role"]
    avatar = "🧑‍💻" if role == "user" else current_avatar
    with st.chat_message(role, avatar=avatar):
        st.markdown(msg["content"])

        # RAG 출처 아코디언 렌더링
        citations = msg.get("citations")
        if citations:
            with st.expander(f"📌 참고한 지식 문서 (출처 {len(citations)}건)", expanded=False):
                for idx, c in enumerate(citations):
                    sim_pct = int(c.get("similarity", 0.0) * 100)
                    st.markdown(
                        f"""
                        <div class="citation-card">
                            <div class="citation-header">
                                <span>📄 [{idx+1}] {c.get('source')}</span>
                                <span style="color: #818CF8;">신뢰도: {sim_pct}%</span>
                            </div>
                            <div style="color: #94A3B8; font-size: 0.8rem; white-space: pre-wrap;">{c.get('content')}</div>
                        </div>
                        """,
                        unsafe_allow_html=True
                    )

        if role == "assistant" and "latency" in msg:
            meta_str = f"⏱️ 응답 시간: **{msg['latency']}초**"
            if msg.get("tokens"):
                tps = round(msg['tokens'] / max(msg['latency'], 0.01), 1)
                meta_str += f" | 생성 토큰: **{msg['tokens']}개** (~{tps} tps)"
            if msg.get("timestamp"):
                meta_str += f" | 🕒 {msg['timestamp']}"
            st.caption(meta_str)


# ===== 10. 사용자 입력 처리 로직 =====
active_input = None
user_chat_input = st.chat_input("교안이나 배포에 대해 질문하세요... (Enter)")

if st.session_state.get("pending_input"):
    active_input = st.session_state["pending_input"]
    st.session_state["pending_input"] = None
elif user_chat_input:
    active_input = user_chat_input

if active_input:
    curr_time_str = datetime.now().strftime("%H:%M:%S")

    # 사용자 메시지 세션에 추가
    st.session_state["chat_history"].append({
        "role": "user",
        "content": active_input,
        "timestamp": curr_time_str,
    })

    with st.chat_message("user", avatar="🧑‍💻"):
        st.markdown(active_input)

    # 챗봇 응답 생성
    with st.chat_message("assistant", avatar=current_avatar):
        start_time = time.time()

        stream_gen, retrieved_citations = stream_chat_api_rag(
            messages=st.session_state["chat_history"],
            system_prompt=active_system_prompt,
            api_key=api_key,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            use_rag=use_rag,
            top_k=top_k,
        )

        full_response = st.write_stream(stream_gen)
        elapsed = round(time.time() - start_time, 2)

        # RAG 출처 표시
        if retrieved_citations:
            with st.expander(f"📌 참고한 지식 문서 (출처 {len(retrieved_citations)}건)", expanded=False):
                for idx, c in enumerate(retrieved_citations):
                    sim_pct = int(c.get("similarity", 0.0) * 100)
                    st.markdown(
                        f"""
                        <div class="citation-card">
                            <div class="citation-header">
                                <span>📄 [{idx+1}] {c.get('source')}</span>
                                <span style="color: #818CF8;">신뢰도: {sim_pct}%</span>
                            </div>
                            <div style="color: #94A3B8; font-size: 0.8rem; white-space: pre-wrap;">{c.get('content')}</div>
                        </div>
                        """,
                        unsafe_allow_html=True
                    )

        if full_response and not full_response.startswith("❌") and not full_response.startswith("🔌"):
            approx_tokens = len(full_response.split()) * 2
            st.session_state["total_tokens_generated"] += approx_tokens
            st.session_state["latency_history"].append(elapsed)

            st.session_state["chat_history"].append({
                "role": "assistant",
                "content": full_response,
                "latency": elapsed,
                "tokens": approx_tokens,
                "timestamp": datetime.now().strftime("%H:%M:%S"),
                "citations": retrieved_citations,
            })
            tps = round(approx_tokens / max(elapsed, 0.01), 1)
            st.caption(f"⏱️ 응답 시간: **{elapsed}초** | 생성 토큰: ~**{approx_tokens}개** (~{tps} tps)")



Overwriting frontend/app_chatbot_v2.py


## 4. 백엔드 서버 기동 및 RAG API 테스트

In [9]:
# FastAPI v2 서버 기동 (LLM + ChromaDB)
serve_in_thread("app.chatbot_api_v2:app", port=8000)

⚠️ 포트 8000를 다른 프로세스가 사용 중입니다.


In [10]:
import requests

# 1. 헬스체크 및 RAG 상태 확인
resp = requests.get("http://localhost:8000/health")
print("헬스체크 응답:", resp.json())

헬스체크 응답: {'status': 'healthy', 'model_name': 'Qwen/Qwen2.5-0.5B-Instruct', 'device': 'cpu', 'rag_ready': True, 'rag_total_chunks': 274, 'version': '2.1.0'}


In [11]:
# 2. RAG 지식 통계 확인
stats_resp = requests.get("http://localhost:8000/rag/stats", headers={"X-API-Key": "test-key-001"})
print("RAG 지식 통계:", stats_resp.json())

RAG 지식 통계: {'success': True, 'user': '사용자A', 'stats': {'total_chunks': 274, 'unique_documents': ['Quest0820.md', 'Quest0818.md', 'Quest0814.md', 'README.md', 'Quest.md', 'Quest0819.md', 'Quest0813.md'], 'doc_count': 7, 'storage_path': 'data/chroma'}}


In [12]:
import json
import urllib.parse
import requests

headers = {"X-API-Key": "test-key-001"}
payload = {
    "messages": [
        {"role": "user", "content": "FastAPI에서 PyTorch 추론할 때 run_in_executor를 사용하는 이유를 코스 문서를 기반으로 알려줘."}
    ],
    "use_rag": True,
    "top_k": 2,
    "max_new_tokens": 120,
    "temperature": 0.7
}

print("🌊 실시간 RAG 스트리밍 수신 중: ", end="", flush=True)
with requests.post("http://localhost:8000/chat/stream", json=payload, headers=headers, stream=True) as r:
    # 출처 헤더 확인
    citations_hdr = r.headers.get("X-Citations")
    for chunk in r.iter_content(chunk_size=None, decode_unicode=True):
        if chunk:
            print(chunk, end="", flush=True)

if citations_hdr:
    citations = json.loads(urllib.parse.unquote(citations_hdr))
    print(f"\n\n📌 인용된 지식 출처 ({len(citations)}건):")
    for c in citations:
        print(f"  - 📄 {c['source']} (유사도: {int(c['similarity']*100)}%)")

🌊 실시간 RAG 스트리밍 수신 중: PyTorch의 `run_in_executor` 함수는 FastAPI에 의해 PyTorch 모델을 푼 후에 실행되는 작업을 처리하는 역할을 하는 함수입니다. 여기서 설명드리겠습니다:

1. **PyTorch 모델 전처리와 푸쉬**: 먼저, PyTorch 모델을 준비합니다. 이를 위해 `torch.compile()` 함수가 호출됩니다. 이 함수는 모델의 전역 변수와 함수를 컴파일하여 학습된 모델을 쉽게 사용하기 위해 사용됩니다.

2. **FastAPI 실행**:

📌 인용된 지식 출처 (2건):
  - 📄 README.md (유사도: 70%)
  - 📄 Quest0820.md (유사도: 69%)


## 5. Streamlit 대시보드 실행

In [13]:
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "frontend/app_chatbot_v2.py",
     "--server.port", "8501",
     "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print("✅ Streamlit v2 RAG 대시보드가 포트 8501에서 실행 중입니다.")
print("👉 브라우저 접속: http://localhost:8501")

✅ Streamlit v2 RAG 대시보드가 포트 8501에서 실행 중입니다.
👉 브라우저 접속: http://localhost:8501


---

## 6. 서비스 고도화 결과 및 주요 개선점 총괄

본 실습(v2)에서는 기존 Day 7의 기본 트랜스포머 챗봇 서빙 구조를 대폭 확장하여, **로컬 경량 LLM(0.5B)의 한계를 극복하고 실무 엔터프라이즈 환경에 부합하는 RAG(검색 증강 생성) 지식 서빙 파이프라인**을 완성했습니다.


### 6.1 Streamlit v2 대시보드 UI 프리뷰

FastAPI 비동기 백엔드 서버와 실시간으로 연동되는 Streamlit v2 챗봇 대시보드의 초기 화면입니다. 추천 질문 칩, 페르소나 프리셋, 생성 파라미터 제어 슬라이더(Temperature, Top-p, Max Tokens), 실시간 세션 통계 메트릭 패널이 통합되어 있습니다.

![AIFFEL Korean LLM Chatbot v2 대시보드 화면](images/chatbot_v2_dashboard.png)


### 6.2 실시간 멀티턴 대화 및 세션 메트릭 실행 결과

사용자의 연속적인 질의("TorchScript vs ONNX 비교" 및 후속 질문)에 대해 이전 대화 맥락(Context)을 완벽히 유지하며 토큰 단위로 실시간 스트리밍 답변을 생성합니다. 좌측 사이드바에서는 대화 턴 수(2턴), 누적 생성 토큰(226개), 평균 지연시간(20.87초) 및 실시간 응답 지연시간 추이 그래프가 자동으로 집계됩니다.

![실시간 멀티턴 대화 및 메트릭 실행 화면](images/chatbot_v2_chat_multiturn.png)


### 6.3 Day 7 기본 버전 vs v2 고도화 버전 주요 개선점 비교

| 비교 항목 | Day 7 기본 버전 | Day 7 v2 고도화 버전 | 주요 개선 효과 |
| :--- | :--- | :--- | :--- |
| **지식 범위 및 신뢰성** | 사전 학습된 일반 상식만 답변 (환각 위험) | **ChromaDB 로컬 벡터 DB RAG 연동** | 코스 교안/코드 기반 팩트 답변 + 인용 출처(Citations) 제공 |
| **추론 응답 방식** | 단발성 배치 추론 (전체 문장 대기) | **FastAPI StreamingResponse + 실시간 스트리밍** | 첫 글자 반응 속도(TTFT) 대폭 단축, 실시간 타이핑 효과 |
| **비동기 서버 안정성** | 동기 추론 시 이벤트 루프 블로킹 가능성 | **ThreadPoolExecutor + run_in_executor 격리** | CPU 집중 추론 중에도 다른 API 요청의 동시 처리 보장 |
| **대화 맥락 관리** | 단순 턴 슬라이싱 (잘림 위험) | **안전한 [User, Assistant] 페어 윈도우 슬라이싱** | 긴 대화에서도 프롬프트 구조 깨짐 없이 컨텍스트 안정적 유지 |
| **프론트엔드 UI/UX** | 기본 텍스트 챗봇 레이아웃 | **프리미엄 다크 글래스모피즘 + 실시간 메트릭 대시보드** | 세션 통계 그래프, 대화 내보내기(MD/JSON), 추천 질문 칩 제공 |
| **시스템 안정성** | Windows 인코딩 충돌 가능성 | **UTF-8 & RFC 헤더 ASCII 인코딩 표준화** | Windows 환경에서도 404/500 에러 없는 견고한 서빙 보장 |


### 6.4 최종 결론 및 서빙 아키텍처 의의

1. **무상태(Stateless) 마이크로서비스 설계**:
   - 서버가 대화 상태를 메모리에 보관하지 않고 클라이언트가 세션을 주도하도록 설계하여, 다중 인스턴스 환경에서 수평 확장(Scale-out)과 로드 밸런싱이 매우 용이합니다.
2. **경량 모델 서빙의 실무 표준 체득**:
   - 대형 GPU 클러스터 없이도 로컬 환경에서 **초경량 LLM(0.5B) + SentenceTransformers 임베딩 + ChromaDB**의 조합을 통해 엔터프라이즈급 사내 지식 검색 서비스를 구현하는 실전 감각을 완성했습니다.
